In [1]:
import pandas as pd

In [2]:
df_train = pd.read_csv("data/final_training_data.csv")

In [15]:
df_train['label'].value_counts()

label
genuine    719334
fault       66996
frozen       2142
spike         522
Name: count, dtype: int64

In [3]:
df_train.columns

Index(['Unnamed: 0', 'date', 'temperature', 'humidity', 'pressure', 'label',
       'cos_hour', 'sin_hour', 'cos_month', 'sin_month', 'temp_gradient',
       'humid_gradient', 'press_gradient', '6hr_gradient_temp',
       '6hr_gradient_press', '6hr_gradient_humid', 'elevation', 'latitude',
       'longitude'],
      dtype='str')

In [4]:
from weather_anamoly.model import LABEL_DECODER, LABEL_ENCODER, FEATURE_COLUMNS
print(LABEL_DECODER, LABEL_ENCODER, FEATURE_COLUMNS)

{0: 'genuine', 1: 'spike', 2: 'frozen', 3: 'fault'} {'genuine': 0, 'spike': 1, 'frozen': 2, 'fault': 3} ['temperature', 'humidity', 'pressure', 'cos_hour', 'sin_hour', 'cos_month', 'sin_month', 'temp_gradient', 'humid_gradient', 'press_gradient', '6hr_gradient_temp', '6hr_gradient_press', '6hr_gradient_humid', 'elevation', 'latitude', 'longitude']


In [16]:
x_train = df_train[FEATURE_COLUMNS]
y_train = df_train['label'].map(LABEL_ENCODER)

In [9]:
df_test = pd.read_csv("data/final_testing_data.csv")

In [17]:
x_test = df_test[FEATURE_COLUMNS]
y_test = df_test['label'].map(LABEL_ENCODER)

In [29]:
from keras_tuner.tuners import Hyperband
import keras_tuner as kt
import tensorflow as tf
from tensorflow.keras.optimizers import Adam

In [69]:
def Build_model(hp):
    model = tf.keras.Sequential()
    num_layers = hp.Int("num_layers", min_value=1, max_value=5, step=1)
    for i in range(num_layers):
        units = hp.Int(f"unit_layer_{i}", min_value=32, max_value=512, step=16)
        model.add(tf.keras.layers.Dense(units=units, activation=hp.Choice(f"activation_layer_{i}", ['relu', 'tanh', 'elu'])))

        if hp.Boolean(f'dropout_layer_{i}'):
            model.add(tf.keras.layers.Dropout(
                hp.Float(f"dropout_rate_{i}", 0.1, 0.5, step=0.1)
            ))
    model.add(tf.keras.layers.Dense(4, activation='softmax'))
    hp_learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])
    model.compile(
        optimizer=Adam(hp_learning_rate),
        loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
        metrics= ['accuracy']
    )
    return model

In [21]:
LABEL_ENCODER

{'genuine': 0, 'spike': 1, 'frozen': 2, 'fault': 3}

In [ ]:
tuner = Hyperband(
    Build_model,
    objective=kt.Objective('val_loss', direction='min'),
    max_epochs=10,
    factor=3,
)from pathlib import Path
from weather_anamoly.utils import Build_model
import json
import pandas as pd

ROOT = Path(__file__).resolve().parent.parent

with open(f"{ROOT}/artifacts/best_hyperparameters.json", "r") as f:
    parameters = json.load(f)

df_train = pd.read_csv("data/final_training_data.csv")

model = Build_model(parameters)
model.fit()
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

In [71]:
tuner.search(
    x_train, y_train,
    validation_data= (x_test, y_test),
    epochs=20,
    batch_size=64,
    shuffle=False,
    class_weight={0:1, 1:2, 2:2, 3:5},
    callbacks=[early_stop]
)

Trial 30 Complete [00h 01m 41s]
val_loss: 1.7228604555130005

Best val_loss So Far: 0.2909100353717804
Total elapsed time: 00h 31m 42s


In [75]:
tuner.results_summary(num_trials=1)

Results summary
Results in ./untitled_project
Showing 1 best trials
Objective(name="val_loss", direction="min")

Trial 0017 summary
Hyperparameters:
num_layers: 4
unit_layer_0: 464
activation_layer_0: relu
dropout_layer_0: False
learning_rate: 0.001
dropout_rate_0: 0.2
unit_layer_1: 480
activation_layer_1: relu
dropout_layer_1: True
unit_layer_2: 176
activation_layer_2: elu
dropout_layer_2: False
unit_layer_3: 272
activation_layer_3: relu
dropout_layer_3: False
unit_layer_4: 512
activation_layer_4: elu
dropout_layer_4: True
dropout_rate_2: 0.30000000000000004
dropout_rate_3: 0.4
dropout_rate_4: 0.30000000000000004
dropout_rate_1: 0.30000000000000004
tuner/epochs: 10
tuner/initial_epoch: 4
tuner/bracket: 2
tuner/round: 2
tuner/trial_id: 0013
Score: 0.2909100353717804


In [80]:
best_params = tuner.get_best_hyperparameters(num_trials=1)[0]
print(best_params['num_layers'])

4


In [78]:
best_trial = tuner.oracle.get_best_trials(num_trials=1)[0]

print(f"Best Trial ID: {best_trial.trial_id}")
print(f"Best Objective Score ({tuner.oracle.objective.name}): {best_trial.score}")
print("\nAll logged metrics for the best trial:")
for metric_name, metric_history in best_trial.metrics.metrics.items():
    print(f"{metric_name}: {metric_history.get_best_value()}")

Best Trial ID: 0017
Best Objective Score (val_loss): 0.2909100353717804

All logged metrics for the best trial:
accuracy: 0.931860089302063
loss: 0.8601183891296387
val_accuracy: 0.9782877564430237
val_loss: 0.2909100353717804


In [84]:
import json

param_dict = best_params.values

with open("/home/Beta/Desktop/coding/weather_anamoly/artifacts/best_hyperparameters.json", "w") as f:
    json.dump(param_dict, f, indent=4)


In [87]:
model = Build_model(best_params)
model.fit(x_train, y_train, batch_size=64, shuffle=False, epochs=3, validation_data=(x_test, y_test))

Epoch 1/3
12329/12329 ━━━━━━━━━━━━━━━━━━━━ 25s 2ms/step - accuracy: 0.9249 - loss: 0.5670 - val_accuracy: 0.9785 - val_loss: 0.2527
Epoch 2/3
12329/12329 ━━━━━━━━━━━━━━━━━━━━ 24s 2ms/step - accuracy: 0.9341 - loss: 0.2809 - val_accuracy: 0.9779 - val_loss: 0.2053
Epoch 3/3
12329/12329 ━━━━━━━━━━━━━━━━━━━━ 24s 2ms/step - accuracy: 0.9387 - loss: 0.2901 - val_accuracy: 0.9832 - val_loss: 0.1496


In [88]:
model.save(filepath="/home/Beta/Desktop/coding/weather_anamoly/artifacts/History_model.keras")